In [1]:
!pip install firebase-admin -q

import torch
import torch.nn as nn
import numpy as np
from collections import deque
import time
import firebase_admin
from firebase_admin import credentials
from firebase_admin import db

# ==========================================
# 1. MODEL DEFINITION
# ==========================================
class LocomotionHybrid(nn.Module):
    def __init__(self, num_classes=7):
        super(LocomotionHybrid, self).__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv1d(6, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.3)
        )
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=0.4)
        self.fc = nn.Sequential(
            nn.Linear(128 * 2, 64), nn.ReLU(),
            nn.BatchNorm1d(64), nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv_layer(x)
        x = x.transpose(1, 2)
        x, _ = self.lstm(x)
        x = torch.mean(x, dim=1)
        return self.fc(x)

# ==========================================
# 2. INITIALIZATION & FIREBASE SETUP
# ==========================================
model = LocomotionHybrid(num_classes=7)
model.load_state_dict(torch.load('/content/best_model_finetuned.pth', map_location=torch.device('cpu')))
model.eval()
print("Model Loaded Successfully in Colab!")

cred = credentials.Certificate("/content/firebase_credentials.json.json")
# If Colab complains it's already initialized, this try/except handles it
try:
    firebase_admin.initialize_app(cred, {
        # ---> CHANGE THIS TO YOUR NEW FIREBASE URL <---
        'databaseURL': 'https://rehab-project-1135b-default-rtdb.firebaseio.com/'
    })
except ValueError:
    pass

prediction_ref = db.reference('/sensors/esp32_device/prediction')

WINDOW_SIZE = 50
imu_buffer = deque(maxlen=WINDOW_SIZE)
CLASS_NAMES = {0: "Sitting", 1: "Standing", 2: "Walking", 3: "Running", 4: "Stairs Up", 5: "Stairs Down", 6: "Falling"}

# ==========================================
# 3. FIREBASE LISTENER (The Loop)
# ==========================================
def imu_update_handler(event):
    data = event.data
    if data and isinstance(data, dict):
        try:
            values = [
                data.get('accel_x', 0), data.get('accel_y', 0), data.get('accel_z', 0),
                data.get('gyro_x', 0), data.get('gyro_y', 0), data.get('gyro_z', 0)
            ]

            imu_buffer.append(values)

            # ---> NEW PROGRESS PRINT <---
            if len(imu_buffer) % 25 == 0:
                print(f"Gathering data... {len(imu_buffer)}/250 samples")

            if len(imu_buffer) == WINDOW_SIZE:
                input_array = np.array(imu_buffer)
                input_tensor = torch.tensor(input_array, dtype=torch.float32).unsqueeze(0)

                with torch.no_grad():
                    output = model(input_tensor)
                    _, predicted_idx = torch.max(output, 1)
                    class_index = predicted_idx.item()

                prediction_ref.set(class_index)
                print(f"\n✅ PREDICTION MADE: {CLASS_NAMES.get(class_index)} | Uploaded to Firebase!")

                for _ in range(25): imu_buffer.popleft()

        except Exception as e:
            print(f"Error processing data: {e}")

imu_ref = db.reference('/sensors/esp32_device/imu')
imu_ref.listen(imu_update_handler)

print("Listening to Firebase for IMU updates from the ESP32... (Press the Stop button to exit)")
while True:
    time.sleep(1)

Model Loaded Successfully in Colab!
Listening to Firebase for IMU updates from the ESP32... (Press the Stop button to exit)
Gathering data... 25/250 samples
Gathering data... 50/250 samples

✅ PREDICTION MADE: Standing | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: Standing | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: Standing | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: Stairs Down | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: Stairs Down | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: Stairs Up | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: Stairs Up | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: Stairs Down | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: Stairs Down | Uploaded to Firebase!
Gathering data... 50/250 samples

✅ PREDICTION MADE: St

KeyboardInterrupt: 